In [0]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if project_root not in sys.path:
    sys.path.append(project_root)

from pyspark.sql.functions import count, count_if, max, min, avg, round
from modules.utils.date import get_months_start_n_months_ago

In [0]:
# Aggregate daily trip summaries by city from enriched bike trip records
# 3. Load latest enriched bike trips from silver layer
# 4. Group trips by city and date to calculate daily statistics
# 5. Aggregate trip counts by bike type, duration metrics, and distance metrics
# 6. Sort by date and city
# 7. Append summaries to gold layer daily summary table
target_month = int(dbutils.widgets.get("months_ago"))
target_month_start = get_months_start_n_months_ago(months_ago=target_month)

In [0]:
df = spark.read.table("bikes.02_silver.trips_enriched").filter(f"started_at >= '{target_month_start}'")

In [0]:
df = df.groupBy(df.city,df.started_at.cast("date").alias("date")).\
    agg(
        count("*").alias("total_trips"),
        count_if(df.rideable_type == "classic_bike").alias("total_classic_bike_trips"),
        count_if(df.rideable_type == "electric_bike").alias("total_electric_bike_trips"),
        round(avg(df.trip_duration_mins),2).alias("avg_trip_duration"),
        max(df.trip_duration_mins).alias("max_trip_duration"),
        min(df.trip_duration_mins).alias("min_trip_duration"),
        round(avg(df.stations_distance),4).alias("avg_stations_distance"),
        round(max(df.stations_distance),4).alias("max_stations_distance"),
        round(min(df.stations_distance),4).alias("min_stations_distance")
    )

In [0]:
df.sort(df.date,df.city).write.mode("append").saveAsTable("bikes.03_gold.daily_trip_summary")